# Agente PySpark + Engenharia de Dados + Iceberg

Referencia completa para engenharia de dados com PySpark, cobrindo fundamentos ate Apache Iceberg.

| Secao | Tema | Nivel |
|-------|------|-------|
| **1** | Fundamentos do PySpark | Basico |
| **2** | Operacoes de Dados | Intermediario |
| **3** | Otimizacao de Performance | Avancado |
| **4** | Gerenciamento de Cluster | Intermediario |
| **5** | Ecossistema Spark | Intermediario |
| **6** | Apache Iceberg | Avancado |
| **7** | Troubleshooting | Referencia |

---

# 1. Fundamentos do PySpark

## 1.1 — O que e o Apache Spark

Motor de processamento distribuido para grandes volumes de dados. Diferente do Hadoop MapReduce, processa **em memoria** (ate 100x mais rapido).

```
+-----------------------+
|     Spark Application |
+-----------+-----------+
|  Driver   | SparkSession / SparkContext
+-----------+-----------+
            |
    +-------+-------+
    |       |       |
+---v-+ +---v-+ +---v-+
|Exec1| |Exec2| |Exec3|  <- Executors nos Workers
|Task | |Task | |Task |
+-----+ +-----+ +-----+
```

- **Driver** — processo principal, cria o plano de execucao
- **Executors** — processos nos workers que executam as tarefas
- **Tasks** — unidade minima de trabalho (1 task por particao)

## 1.2 — SparkSession

Ponto de entrada unico para toda interacao com Spark (substituiu SparkContext, SQLContext, HiveContext).

In [ ]:
from pyspark.sql import SparkSession

# Criacao basica (usa spark-defaults.conf do cluster)
spark = SparkSession.builder \
    .appName("agente-pyspark") \
    .getOrCreate()

print(f"Spark version: {spark.version}")
print(f"Master: {spark.sparkContext.master}")
print(f"App ID: {spark.sparkContext.applicationId}")
print(f"Driver host: {spark.conf.get('spark.driver.host')}")
print(f"Cores max: {spark.conf.get('spark.cores.max')}")
print(f"Executor memory: {spark.conf.get('spark.executor.memory')}")

**`getOrCreate()`** — se ja existe uma sessao com mesmo appName, reutiliza. Se nao, cria nova.

**Configs podem ser definidas em 3 niveis (ordem de prioridade):**

| Prioridade | Onde | Exemplo |
|------------|------|--------|
| 1 (maior) | Codigo `.config()` | `builder.config('spark.cores.max', '4')` |
| 2 | spark-submit flags | `--conf spark.cores.max=4` |
| 3 (menor) | spark-defaults.conf | `spark.cores.max  4` |

## 1.3 — RDD vs DataFrame vs Dataset

| Aspecto | RDD | DataFrame | Dataset (Scala/Java) |
|---------|-----|-----------|---------------------|
| Tipagem | Nao tipado | Schema (colunas tipadas) | Tipado em compile-time |
| Otimizacao | Nenhuma (manual) | Catalyst + Tungsten | Catalyst + Tungsten |
| API | Funcional (map, filter) | Declarativa (select, where) | Ambas |
| Performance | Mais lento | Otimizado | Otimizado |
| Uso em Python | Sim | **Sim (preferido)** | Nao disponivel |

**Regra pratica:** Em PySpark, **sempre use DataFrame**. RDD so para operacoes de baixo nivel que DataFrame nao suporta.

In [ ]:
# RDD — baixo nivel (evitar)
rdd = spark.sparkContext.parallelize([1, 2, 3, 4, 5])
print(f"RDD soma: {rdd.reduce(lambda a, b: a + b)}")

# DataFrame — alto nivel (preferido)
df = spark.createDataFrame(
    [(1, "Ana", 5000.0), (2, "Bruno", 6000.0), (3, "Carlos", 4500.0)],
    ["id", "nome", "salario"]
)
df.show()
df.printSchema()

## 1.4 — Lazy Evaluation: Transformacoes vs Acoes

Spark **nao executa nada** ate que uma **acao** seja chamada. Transformacoes apenas montam o plano (DAG).

| Transformacoes (lazy) | Acoes (executam) |
|----------------------|------------------|
| `select()`, `filter()` | `show()`, `collect()` |
| `withColumn()`, `drop()` | `count()`, `first()` |
| `groupBy()`, `join()` | `write.save()` |
| `orderBy()`, `distinct()` | `take(n)`, `toPandas()` |
| `union()`, `repartition()` | `foreach()`, `reduce()` |

In [ ]:
from pyspark.sql import functions as F

# Nada executa aqui (lazy) — so monta o plano
resultado = df \
    .filter(F.col("salario") > 4500) \
    .withColumn("bonus", F.col("salario") * 0.1) \
    .select("nome", "salario", "bonus")

# Agora sim — acao dispara a execucao
resultado.show()
print(f"Registros: {resultado.count()}")

## 1.5 — Anatomia de um Job

```
Application (1 SparkSession)
  └── Job (1 por acao: show, count, write)
       └── Stage (separado por shuffles)
            └── Task (1 por particao)
```

**Exemplo:** `df.groupBy('dept').count().show()` gera:
- **Job 1** — o `show()` dispara
- **Stage 1** — ler dados e fazer map-side aggregation (antes do shuffle)
- **Stage 2** — shuffle + reduce-side aggregation (apos o shuffle)
- **Tasks** — 1 por particao de entrada

Veja no Spark UI (http://localhost:8090) os jobs, stages e tasks de cada aplicacao.

## 1.6 — Criacao de DataFrames

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType
from datetime import date

# 1) A partir de lista de tuplas
df1 = spark.createDataFrame(
    [(1, "Ana"), (2, "Bruno")],
    ["id", "nome"]
)

# 2) Com schema explicito (recomendado para producao)
schema = StructType([
    StructField("id", IntegerType(), nullable=False),
    StructField("nome", StringType(), nullable=False),
    StructField("salario", DoubleType(), nullable=True),
    StructField("data_admissao", DateType(), nullable=True)
])

df2 = spark.createDataFrame([
    (1, "Ana", 5000.0, date(2023, 1, 15)),
    (2, "Bruno", 6000.0, date(2023, 3, 20)),
    (3, "Carlos", None, date(2024, 6, 1))
], schema=schema)

df2.show()
df2.printSchema()

# 3) A partir de Pandas
import pandas as pd
pdf = pd.DataFrame({"x": [1, 2, 3], "y": ["a", "b", "c"]})
df3 = spark.createDataFrame(pdf)
df3.show()

# 4) A partir de range
df4 = spark.range(0, 1000000, 1, numPartitions=10)
print(f"Range: {df4.count()} registros, {df4.rdd.getNumPartitions()} particoes")

---

# 2. Operacoes de Dados

## 2.1 — Leitura e Escrita de Dados

In [ ]:
# === LEITURA ===

# CSV
# df = spark.read.csv("s3a://landing/dados.csv", header=True, inferSchema=True, sep=";")

# JSON
# df = spark.read.json("s3a://landing/dados.json")
# df = spark.read.json("s3a://landing/*.json")  # wildcard

# Parquet (formato colunar, recomendado)
# df = spark.read.parquet("s3a://bronze/tabela/")

# ORC
# df = spark.read.orc("s3a://bronze/tabela_orc/")

# Delta Lake
# df = spark.read.format("delta").load("s3a://bronze/tabela_delta/")

# JDBC (banco relacional)
# df = spark.read.jdbc(
#     url="jdbc:postgresql://host:5432/db",
#     table="schema.tabela",
#     properties={"user": "usr", "password": "pwd"}
# )

# === ESCRITA ===

# Modos de escrita:
# - "overwrite"  — substitui tudo
# - "append"     — adiciona ao existente
# - "ignore"     — nao faz nada se ja existir
# - "error"      — erro se ja existir (padrao)

# Parquet com particionamento
# df.write.mode("overwrite") \
#     .partitionBy("ano", "mes") \
#     .parquet("s3a://bronze/tabela/")

# Delta Lake
# df.write.format("delta").mode("overwrite").save("s3a://bronze/tabela_delta/")

print("Exemplos de leitura/escrita (descomente para usar com MinIO)")

### Comparacao de formatos

| Formato | Tipo | Compressao | Schema | ACID | Melhor para |
|---------|------|-----------|--------|------|-------------|
| CSV | Linha | Nenhuma | Nao | Nao | Importacao simples |
| JSON | Semi-estruturado | Nenhuma | Inferido | Nao | APIs, logs |
| Parquet | **Colunar** | Snappy/Gzip | **Embutido** | Nao | Data lake (leitura) |
| ORC | Colunar | Zlib/Snappy | Embutido | Nao | Hive ecosystem |
| Delta | Colunar (Parquet+log) | Snappy | Embutido | **Sim** | Data lake (ACID) |
| **Iceberg** | Colunar (Parquet+metadata) | Snappy | Embutido | **Sim** | Data lake (evolucao) |

## 2.2 — Schema: Definicao e Manipulacao

In [ ]:
from pyspark.sql.types import *

# Schema completo com tipos aninhados
schema_completo = StructType([
    StructField("id", IntegerType(), False),
    StructField("nome", StringType(), False),
    StructField("salario", DoubleType(), True),
    StructField("ativo", BooleanType(), True),
    StructField("data_nasc", DateType(), True),
    StructField("timestamp_reg", TimestampType(), True),
    # Struct aninhado
    StructField("endereco", StructType([
        StructField("rua", StringType(), True),
        StructField("cidade", StringType(), True),
        StructField("uf", StringType(), True)
    ]), True),
    # Array
    StructField("telefones", ArrayType(StringType()), True),
    # Map
    StructField("metadata", MapType(StringType(), StringType()), True)
])

# Tipos disponiveis
tipos = [
    ("StringType", "Texto"),
    ("IntegerType / LongType", "Inteiro 32/64 bits"),
    ("FloatType / DoubleType", "Decimal 32/64 bits"),
    ("DecimalType(p, s)", "Decimal exato (financeiro)"),
    ("BooleanType", "True/False"),
    ("DateType", "Data (sem hora)"),
    ("TimestampType", "Data + hora"),
    ("BinaryType", "Bytes"),
    ("ArrayType(T)", "Lista de T"),
    ("MapType(K, V)", "Dicionario K->V"),
    ("StructType", "Objeto aninhado")
]

spark.createDataFrame(tipos, ["Tipo", "Descricao"]).show(truncate=False)

# Cast de tipos
df_cast = df.withColumn("salario_int", F.col("salario").cast(IntegerType()))
df_cast.printSchema()

## 2.3 — Select, Colunas e Transformacoes

In [ ]:
# Dados de exemplo para operacoes
funcionarios = spark.createDataFrame([
    (1, "Ana Silva", "TI", 8500.0, date(2020, 3, 15)),
    (2, "Bruno Costa", "RH", 6200.0, date(2019, 7, 1)),
    (3, "Carlos Lima", "TI", 9100.0, date(2021, 1, 10)),
    (4, "Diana Souza", "Vendas", 7300.0, date(2018, 11, 20)),
    (5, "Eduardo Reis", "TI", 8800.0, date(2022, 5, 5)),
    (6, "Fernanda Dias", "RH", 6500.0, date(2020, 9, 12)),
    (7, "Gabriel Nunes", "Vendas", 7800.0, date(2021, 4, 3)),
    (8, "Helena Rocha", "TI", 9500.0, date(2017, 2, 28)),
    (9, "Igor Alves", "Vendas", None, date(2023, 8, 15)),
    (10, "Julia Mota", "RH", 5800.0, None)
], ["id", "nome", "departamento", "salario", "data_admissao"])

funcionarios.show(truncate=False)

In [ ]:
# Select — formas equivalentes
funcionarios.select("nome", "salario").show(3)
funcionarios.select(F.col("nome"), F.col("salario")).show(3)
funcionarios.select(funcionarios.nome, funcionarios.salario).show(3)

# withColumn — adiciona ou substitui coluna
funcionarios \
    .withColumn("salario_anual", F.col("salario") * 13) \
    .withColumn("departamento_upper", F.upper(F.col("departamento"))) \
    .select("nome", "salario", "salario_anual", "departamento_upper") \
    .show(3)

# Rename
funcionarios.withColumnRenamed("nome", "nome_completo").columns

# Drop
funcionarios.drop("id").columns

## 2.4 — Filtragem

In [ ]:
# filter e where sao identicos
funcionarios.filter(F.col("salario") > 8000).show()

# Combinando condicoes (& = AND, | = OR, ~ = NOT)
funcionarios.filter(
    (F.col("departamento") == "TI") & (F.col("salario") > 8500)
).show()

# between
funcionarios.filter(F.col("salario").between(6000, 8000)).show()

# isin
funcionarios.filter(F.col("departamento").isin("TI", "RH")).show()

# isNull / isNotNull
funcionarios.filter(F.col("salario").isNull()).show()
funcionarios.filter(F.col("salario").isNotNull()).show()

# like / rlike (regex)
funcionarios.filter(F.col("nome").like("%Silva%")).show()
funcionarios.filter(F.col("nome").rlike("^[A-E]")).show()  # nomes que comecam com A-E

## 2.5 — Joins

```
INNER       LEFT        RIGHT       FULL         CROSS
 +--+      +----+        +----+    +------+    Cada linha
 |AB|      |A|AB|        |AB|B|    |A|AB|B|    de A com
 +--+      +----+        +----+    +------+    cada de B

LEFT SEMI     LEFT ANTI
 +--+          +--+
 |A |          |A |
 +--+          +--+
 (A que tem B)  (A que NAO tem B)
```

In [ ]:
# Tabela de departamentos
departamentos = spark.createDataFrame([
    ("TI", "Tecnologia da Informacao", 3),
    ("RH", "Recursos Humanos", 2),
    ("Vendas", "Comercial", 1),
    ("Financeiro", "Financeiro", 4)  # sem funcionarios
], ["sigla", "nome_completo", "andar"])

# INNER JOIN — so matches
funcionarios.join(departamentos, funcionarios.departamento == departamentos.sigla, "inner") \
    .select("nome", "departamento", "nome_completo", "andar").show(3)

# LEFT JOIN — todos da esquerda
funcionarios.join(departamentos, funcionarios.departamento == departamentos.sigla, "left") \
    .select("nome", "departamento", "nome_completo").show(3)

# RIGHT JOIN — todos da direita (mostra Financeiro sem funcionarios)
funcionarios.join(departamentos, funcionarios.departamento == departamentos.sigla, "right") \
    .select("nome", "sigla", "nome_completo").show()

# LEFT SEMI — funcionarios que TEM departamento (sem colunas da direita)
funcionarios.join(departamentos, funcionarios.departamento == departamentos.sigla, "left_semi").show(3)

# LEFT ANTI — funcionarios que NAO tem departamento
funcionarios.join(departamentos, funcionarios.departamento == departamentos.sigla, "left_anti").show()

## 2.6 — Agregacoes

In [ ]:
# GroupBy + funcoes de agregacao
funcionarios.groupBy("departamento").agg(
    F.count("*").alias("total"),
    F.avg("salario").alias("media_salario"),
    F.min("salario").alias("menor_salario"),
    F.max("salario").alias("maior_salario"),
    F.sum("salario").alias("folha_total"),
    F.stddev("salario").alias("desvio_padrao"),
    F.collect_list("nome").alias("funcionarios")
).show(truncate=False)

# Pivot — transforma valores de linha em colunas
vendas = spark.createDataFrame([
    ("Ana", "Jan", 100), ("Ana", "Fev", 150), ("Ana", "Mar", 120),
    ("Bruno", "Jan", 200), ("Bruno", "Fev", 180), ("Bruno", "Mar", 220)
], ["vendedor", "mes", "valor"])

vendas.groupBy("vendedor").pivot("mes", ["Jan", "Fev", "Mar"]).sum("valor").show()

## 2.7 — Window Functions

In [ ]:
from pyspark.sql.window import Window

# Janela por departamento, ordenada por salario descendente
w = Window.partitionBy("departamento").orderBy(F.col("salario").desc())

funcionarios.filter(F.col("salario").isNotNull()) \
    .withColumn("rank", F.rank().over(w)) \
    .withColumn("dense_rank", F.dense_rank().over(w)) \
    .withColumn("row_number", F.row_number().over(w)) \
    .select("nome", "departamento", "salario", "rank", "dense_rank", "row_number") \
    .show()

# Lead / Lag — valor da proxima/anterior linha
w2 = Window.partitionBy("departamento").orderBy("data_admissao")

funcionarios.filter(F.col("data_admissao").isNotNull()) \
    .withColumn("proximo", F.lead("nome").over(w2)) \
    .withColumn("anterior", F.lag("nome").over(w2)) \
    .select("nome", "departamento", "data_admissao", "anterior", "proximo") \
    .show(truncate=False)

# Soma acumulada
w3 = Window.partitionBy("departamento").orderBy("salario").rowsBetween(Window.unboundedPreceding, Window.currentRow)

funcionarios.filter(F.col("salario").isNotNull()) \
    .withColumn("soma_acumulada", F.sum("salario").over(w3)) \
    .select("nome", "departamento", "salario", "soma_acumulada") \
    .show()

## 2.8 — Strings, Datas e Nulls

In [ ]:
# === STRINGS ===
funcionarios.select(
    "nome",
    F.upper("nome").alias("upper"),
    F.lower("nome").alias("lower"),
    F.length("nome").alias("tamanho"),
    F.split("nome", " ")[0].alias("primeiro_nome"),
    F.substring("nome", 1, 3).alias("primeiras_3"),
    F.trim(F.lit("  texto  ")).alias("trim"),
    F.regexp_replace("nome", "[aeiou]", "*").alias("sem_vogais")
).show(3, truncate=False)

# === DATAS ===
funcionarios.filter(F.col("data_admissao").isNotNull()).select(
    "nome", "data_admissao",
    F.year("data_admissao").alias("ano"),
    F.month("data_admissao").alias("mes"),
    F.dayofmonth("data_admissao").alias("dia"),
    F.datediff(F.current_date(), F.col("data_admissao")).alias("dias_empresa"),
    F.months_between(F.current_date(), F.col("data_admissao")).cast("int").alias("meses_empresa"),
    F.date_format("data_admissao", "dd/MM/yyyy").alias("formato_br")
).show(3, truncate=False)

# === NULLS ===
funcionarios.select(
    "nome", "salario",
    F.coalesce(F.col("salario"), F.lit(0.0)).alias("salario_sem_null"),
    F.when(F.col("salario").isNull(), "Sem salario").otherwise("OK").alias("status")
).show()

## 2.9 — SQL Direto

In [ ]:
# Registrar DataFrame como view temporaria
funcionarios.createOrReplaceTempView("func")

# Consulta SQL pura
spark.sql("""
    SELECT 
        departamento,
        COUNT(*) as total,
        ROUND(AVG(salario), 2) as media,
        MAX(salario) as maior
    FROM func
    WHERE salario IS NOT NULL
    GROUP BY departamento
    ORDER BY media DESC
""").show()

# Subquery
spark.sql("""
    SELECT nome, salario, departamento
    FROM func
    WHERE salario > (SELECT AVG(salario) FROM func WHERE salario IS NOT NULL)
""").show()

---

# 3. Otimizacao de Performance

## 3.1 — Particionamento

Particoes sao a unidade de paralelismo do Spark. Cada particao e processada por 1 task em 1 core.

```
DataFrame (6 particoes)
  [P0] [P1] [P2] [P3] [P4] [P5]
    |    |    |    |    |    |
  Task  Task Task Task Task Task  <- 1 task por particao
```

**Regras praticas:**
- Particoes ideais: **2-4x o numero de cores** do cluster
- Tamanho ideal por particao: **128 MB - 256 MB**
- Poucos particoes = subutiliza cores (lento)
- Muitas particoes = overhead de scheduling (lento tambem)

In [ ]:
# Verificar particoes atuais
big_df = spark.range(0, 1000000)
print(f"Particoes padrao: {big_df.rdd.getNumPartitions()}")

# repartition(n) — redistribui com SHUFFLE (caro, mas equilibrado)
df_repart = big_df.repartition(6)
print(f"Apos repartition(6): {df_repart.rdd.getNumPartitions()}")

# coalesce(n) — reduz SEM shuffle (barato, mas pode desbalancear)
df_coal = big_df.coalesce(2)
print(f"Apos coalesce(2): {df_coal.rdd.getNumPartitions()}")

# repartition por coluna — agrupa dados da mesma chave na mesma particao
df_by_col = funcionarios.repartition("departamento")
print(f"Repartition por coluna: {df_by_col.rdd.getNumPartitions()}")

# partitionBy na ESCRITA — cria diretorios por valor da coluna
# df.write.partitionBy("ano", "mes").parquet("s3a://bronze/vendas/")
# Resultado no storage:
#   vendas/ano=2024/mes=01/part-00000.parquet
#   vendas/ano=2024/mes=02/part-00000.parquet

print("\nQuando usar qual:")
print("repartition(n)    -> aumentar ou redistribuir particoes (antes de join pesado)")
print("coalesce(n)       -> reduzir particoes (antes de write, evita small files)")
print("partitionBy(col)  -> escrita particionada (filtros rapidos no storage)")

## 3.2 — Cache e Persist

Armazena um DataFrame em memoria/disco para reutilizacao. Evita recomputar o mesmo DAG multiplas vezes.

| Nivel | Onde armazena | Quando usar |
|-------|--------------|-------------|
| `MEMORY_ONLY` | Memoria (padrao do cache) | DataFrame cabe na memoria |
| `MEMORY_AND_DISK` | Memoria + disco (spillover) | DataFrame grande, reutilizado |
| `DISK_ONLY` | So disco | DataFrame muito grande |
| `MEMORY_ONLY_SER` | Memoria serializado | Economiza memoria (mais CPU) |
| `OFF_HEAP` | Memoria fora da JVM | Evita GC pauses |

**Regra:** cache so compensa se o DataFrame e reutilizado **2+ vezes**. Cache desnecessario desperdica memoria.

In [ ]:
from pyspark import StorageLevel

# cache() = persist(MEMORY_AND_DISK) no Spark 3.x
df_cached = funcionarios.filter(F.col("salario").isNotNull()).cache()

# Primeira acao materializa o cache
print(f"Count: {df_cached.count()}")

# Segunda acao usa o cache (rapido)
df_cached.groupBy("departamento").avg("salario").show()
df_cached.filter(F.col("salario") > 8000).show()

# Verificar se esta em cache
print(f"Em cache: {df_cached.is_cached}")

# Liberar cache quando nao precisa mais
df_cached.unpersist()
print(f"Apos unpersist: {df_cached.is_cached}")

# persist com nivel especifico
# df.persist(StorageLevel.MEMORY_AND_DISK)
# df.persist(StorageLevel.DISK_ONLY)

## 3.3 — Broadcast Joins

Quando uma tabela e **pequena** (< 10 MB padrao), Spark pode enviar uma copia para cada executor, evitando shuffle.

```
SEM broadcast (shuffle join):         COM broadcast:
  [Part A0] ----shuffle---> [Join]     [Part A0] + [B completo] -> [Join]
  [Part A1] ----shuffle---> [Join]     [Part A1] + [B completo] -> [Join]
  [Part B0] ----shuffle--->            (sem shuffle!)
  [Part B1] ----shuffle--->
```

In [ ]:
# Broadcast explicito — forca envio da tabela pequena
resultado = funcionarios.join(
    F.broadcast(departamentos),
    funcionarios.departamento == departamentos.sigla
)
resultado.select("nome", "departamento", "andar").show(3)

# Verificar no plano de execucao
resultado.explain()
# Deve mostrar "BroadcastHashJoin" em vez de "SortMergeJoin"

# Configurar threshold automatico (padrao: 10 MB)
# spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "50m")  # aumentar para 50 MB
# spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")   # desabilitar auto-broadcast

## 3.4 — Execution Plans e Catalyst Optimizer

O Catalyst Optimizer transforma sua query em 4 fases:

```
Codigo PySpark/SQL
       ↓
[1] Unresolved Logical Plan  (o que voce escreveu)
       ↓
[2] Analyzed Logical Plan    (resolve nomes de colunas/tabelas)
       ↓
[3] Optimized Logical Plan   (aplica otimizacoes: predicate pushdown, column pruning)
       ↓
[4] Physical Plan            (escolhe algoritmos: SortMerge vs Broadcast, scan vs filter)
       ↓
    Executar
```

In [ ]:
# explain() — mostra o Physical Plan
query = funcionarios \
    .filter(F.col("departamento") == "TI") \
    .filter(F.col("salario") > 8000) \
    .select("nome", "salario")

# Plano simples
query.explain()

# Plano completo (todas as fases)
query.explain(mode="extended")

# O que procurar no plano:
# - "Scan" -> como os dados sao lidos
# - "Filter" -> filtros aplicados (pushdown = bom)
# - "BroadcastHashJoin" -> join eficiente (melhor que SortMergeJoin)
# - "Exchange" -> shuffle (caro, minimizar)
# - "Sort" -> ordenacao (caro para dados grandes)

## 3.5 — AQE (Adaptive Query Execution)

Habilitado por padrao no Spark 3.x. Otimiza em **runtime** (nao so em compile-time).

| Feature | O que faz | Config |
|---------|----------|--------|
| Coalesce Shuffle Partitions | Junta particoes pequenas pos-shuffle | `spark.sql.adaptive.enabled` (true) |
| Skew Join Optimization | Divide particoes desbalanceadas | `spark.sql.adaptive.skewJoin.enabled` (true) |
| Dynamic Partition Pruning | Filtra particoes em runtime | `spark.sql.optimizer.dynamicPartitionPruning.enabled` |

## 3.6 — Salting para Data Skew

Quando uma chave de join/groupBy tem valores muito desbalanceados (ex: 90% dos registros com `pais = "Brasil"`), o Spark concentra tudo em 1 particao.

In [ ]:
import random

# Simular dados com skew — 80% dos registros sao "SP"
dados_skew = [(random.choice(["SP"]*8 + ["RJ", "MG"]), i) for i in range(10000)]
df_skew = spark.createDataFrame(dados_skew, ["estado", "valor"])

df_skew.groupBy("estado").count().show()

# Tecnica: SALTING — adicionar sufixo aleatorio a chave
num_salts = 5
df_salted = df_skew.withColumn("salt", (F.rand() * num_salts).cast("int")) \
    .withColumn("estado_salt", F.concat("estado", F.lit("_"), F.col("salt")))

# Agora a agregacao distribui melhor
df_result = df_salted.groupBy("estado_salt").agg(
    F.count("*").alias("contagem"),
    F.avg("valor").alias("media")
)

# Remover o salt do resultado final
df_final = df_result.withColumn("estado", F.split("estado_salt", "_")[0]) \
    .groupBy("estado").agg(
        F.sum("contagem").alias("total"),
        F.avg("media").alias("media_geral")
    )
df_final.show()

## 3.7 — Configs de Performance Essenciais

| Config | Padrao | Recomendacao |
|--------|--------|-------------|
| `spark.sql.shuffle.partitions` | 200 | Ajustar ao tamanho dos dados (10-2000) |
| `spark.sql.autoBroadcastJoinThreshold` | 10m | Aumentar se tabelas de lookup > 10 MB |
| `spark.sql.adaptive.enabled` | true | Manter habilitado |
| `spark.default.parallelism` | 2x cores | 2-4x total de cores do cluster |
| `spark.sql.files.maxPartitionBytes` | 128m | Tamanho maximo por particao na leitura |
| `spark.serializer` | Java | Usar `org.apache.spark.serializer.KryoSerializer` |
| `spark.memory.fraction` | 0.6 | % da heap para execucao+storage |

---

# 4. Gerenciamento de Cluster e Deployment

## 4.1 — Modos de Cluster

| Modo | Gerenciador | Quando usar |
|------|------------|-------------|
| **Local** | Nenhum (JVM unica) | Desenvolvimento, testes |
| **Standalone** | Spark nativo | Clusters dedicados Spark (nossa stack) |
| **YARN** | Hadoop YARN | Clusters Hadoop existentes |
| **Kubernetes** | K8s | Infraestrutura cloud-native |
| **Mesos** | Apache Mesos | Legado (descontinuado) |

```
Nossa stack:
  Modo: Standalone
  Master: spark://spark-master:7077
  Workers: 3 (2 cores, 2.5 GB cada)
  Total: 6 cores, 7.5 GB
```

## 4.2 — spark-submit

```bash
# Sintaxe basica
spark-submit \\
    --master spark://spark-master:7077 \\
    --deploy-mode client \\
    --driver-memory 1g \\
    --executor-memory 2g \\
    --executor-cores 2 \\
    --num-executors 3 \\
    --conf spark.sql.shuffle.partitions=100 \\
    --jars /path/extra.jar \\
    --py-files utils.py \\
    meu_script.py arg1 arg2
```

| Parametro | Descricao |
|-----------|----------|
| `--master` | URL do cluster |
| `--deploy-mode` | `client` (driver local) ou `cluster` (driver no cluster) |
| `--driver-memory` | Memoria do driver |
| `--executor-memory` | Memoria por executor |
| `--executor-cores` | Cores por executor |
| `--num-executors` | Numero de executors (YARN/K8s) |
| `--conf` | Configs Spark avulsas |
| `--jars` | JARs extras (drivers JDBC, etc) |
| `--py-files` | Modulos Python extras |

## 4.3 — Deploy Mode: Client vs Cluster

| Aspecto | Client | Cluster |
|---------|--------|--------|
| Driver roda onde | Na maquina que submeteu | Dentro do cluster |
| Logs | No terminal do usuario | No cluster (precisa coletar) |
| Conexao de rede | Driver precisa acessar workers | Tudo interno |
| Uso tipico | Jupyter, desenvolvimento | Producao, jobs agendados |

**Jupyter sempre usa `client`** — o driver roda no container Jupyter.

## 4.4 — Sizing de Executors (regra pratica)

```
Cluster: 3 nodes, 16 cores e 64 GB cada (48 cores, 192 GB total)

Reservar para OS/YARN: 1 core + 1 GB por node

Disponivel: 15 cores, 63 GB por node

Opcao 1 — Fat executors:
  3 executors (1 por node), 15 cores, ~60 GB cada
  Problema: GC pesado com muita memoria

Opcao 2 — Thin executors (recomendado):
  15 executors, 3 cores, ~12 GB cada (5 por node)
  Melhor paralelismo, GC mais leve

Opcao 3 — Balanced:
  9 executors, 5 cores, ~20 GB cada (3 por node)
  Bom equilibrio para maioria dos workloads
```

**Regras:**
- Max 5 cores por executor (evita thrashing de IO)
- Reservar 10% da memoria para overhead (`spark.executor.memoryOverhead`)
- Deixar 1 core + 1 GB para o OS em cada node

In [ ]:
# Ver configuracao atual do cluster
print("=== Configuracao da SparkSession ===")
configs = [
    "spark.master", "spark.driver.host", "spark.driver.memory",
    "spark.executor.memory", "spark.cores.max",
    "spark.sql.adaptive.enabled", "spark.sql.shuffle.partitions"
]
for c in configs:
    try:
        print(f"  {c} = {spark.conf.get(c)}")
    except:
        print(f"  {c} = (nao definido)")

# Ver executors ativos
sc = spark.sparkContext
print(f"\n=== Cluster ===")
print(f"  App Name: {sc.appName}")
print(f"  Master: {sc.master}")
print(f"  Default Parallelism: {sc.defaultParallelism}")

---

# 5. Ecossistema Spark

## 5.1 — Structured Streaming

Processamento de dados em tempo real usando a mesma API de DataFrames.

```
Fonte (Kafka, Arquivo, Socket)
       ↓
  Input Table (unbounded)
       ↓
  Query (select, filter, groupBy)
       ↓
  Result Table
       ↓
  Sink (Kafka, Arquivo, Console)
```

| Trigger | Comportamento |
|---------|-------------|
| Default (micro-batch) | Processa assim que o batch anterior termina |
| `processingTime='10 seconds'` | A cada 10 segundos |
| `once=True` | Uma vez e para (batch incremental) |
| `availableNow=True` | Processa tudo disponivel e para |
| `continuous='1 second'` | Modo continuo (experimental, latencia <ms) |

In [ ]:
# Exemplo: Streaming de arquivos JSON (monitora diretorio)
# stream_df = spark.readStream \
#     .schema(schema) \
#     .json("s3a://landing/stream/")
#
# query = stream_df \
#     .groupBy("departamento") \
#     .count() \
#     .writeStream \
#     .outputMode("complete") \
#     .format("console") \
#     .trigger(processingTime="30 seconds") \
#     .start()
#
# query.awaitTermination()

# Exemplo: Streaming para Parquet (append only)
# query = stream_df \
#     .writeStream \
#     .outputMode("append") \
#     .format("parquet") \
#     .option("path", "s3a://bronze/stream_output/") \
#     .option("checkpointLocation", "s3a://bronze/checkpoints/") \
#     .trigger(availableNow=True) \
#     .start()

# Output Modes:
# - "append"   -> so novas linhas (sem agregacao)
# - "complete" -> tabela inteira recalculada (com agregacao)
# - "update"   -> so linhas que mudaram

print("Exemplos de Structured Streaming (descomente para usar)")

## 5.2 — UDFs (User Defined Functions)

| Tipo | Performance | Serializacao | Quando usar |
|------|------------|-------------|-------------|
| Python UDF | Lenta (JVM ↔ Python) | Pickle | Ultima opcao |
| Pandas UDF (Arrow) | **Rapida** (vetorizada) | Apache Arrow | Funcoes complexas |
| Funcoes built-in (F.) | **Mais rapida** | Nenhuma (JVM nativa) | Sempre que possivel |

**Regra:** Prefira funcoes built-in (`F.col`, `F.when`, etc). So use UDF quando nao existir built-in equivalente.

In [ ]:
from pyspark.sql.functions import udf, pandas_udf
from pyspark.sql.types import StringType
import pandas as pd

# 1) Python UDF (lenta — evitar)
@udf(returnType=StringType())
def classificar_salario(salario):
    if salario is None:
        return "N/A"
    elif salario > 8000:
        return "Senior"
    elif salario > 6000:
        return "Pleno"
    else:
        return "Junior"

funcionarios.withColumn("nivel", classificar_salario("salario")).show()

# 2) Pandas UDF (rapida — vetorizada via Arrow)
@pandas_udf(StringType())
def classificar_salario_pandas(salario: pd.Series) -> pd.Series:
    return salario.apply(
        lambda s: "N/A" if pd.isna(s) else "Senior" if s > 8000 else "Pleno" if s > 6000 else "Junior"
    )

funcionarios.withColumn("nivel", classificar_salario_pandas("salario")).show()

# 3) Melhor opcao: built-in (sem UDF)
funcionarios.withColumn("nivel",
    F.when(F.col("salario").isNull(), "N/A")
     .when(F.col("salario") > 8000, "Senior")
     .when(F.col("salario") > 6000, "Pleno")
     .otherwise("Junior")
).show()

## 5.3 — Delta Lake (presente na stack)

Delta Lake adiciona ACID transactions sobre Parquet. Ja configurado nesta stack com JARs + MinIO.

| Feature | Parquet | Delta Lake |
|---------|---------|-----------|
| ACID | Nao | Sim |
| Time Travel | Nao | Sim (versoes) |
| Schema Enforcement | Nao | Sim |
| MERGE (upsert) | Nao | Sim |
| UPDATE/DELETE | Nao | Sim |
| Compaction | Manual | Automatico (OPTIMIZE) |

In [ ]:
# Para usar Delta Lake, precisa destas 3 configs na SparkSession:
# .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
# .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
# .config("spark.delta.logStore.class", "org.apache.spark.sql.delta.storage.S3SingleDriverLogStore")

# Exemplo de MERGE (upsert):
# from delta.tables import DeltaTable
#
# delta_table = DeltaTable.forPath(spark, "s3a://bronze/device/delta")
#
# delta_table.alias("target").merge(
#     novos_dados.alias("source"),
#     "target.id = source.id"
# ).whenMatchedUpdateAll() \
#  .whenNotMatchedInsertAll() \
#  .execute()

# Time Travel:
# df_v0 = spark.read.format("delta").option("versionAsOf", 0).load("s3a://bronze/device/delta")
# df_ts = spark.read.format("delta").option("timestampAsOf", "2024-01-01").load("s3a://bronze/device/delta")

print("Delta Lake: ver notebook medallion-pipeline.ipynb para exemplo completo")

---

# 6. Apache Iceberg

## 6.1 — O que e o Apache Iceberg

Formato de tabela aberto para data lakes analiticos. Diferente de formatos de arquivo (Parquet, ORC), Iceberg e um **formato de tabela** — uma camada de metadados sobre os arquivos de dados.

```
                 +---------------------------+
                 |      Iceberg Table        |
                 +---------------------------+
                 |  Catalog (onde registrar)  |
                 +---------------------------+
                 |  Metadata (JSON files)     |  <- Schema, partitions, snapshots
                 +---------------------------+
                 |  Manifest List (.avro)     |  <- Lista de manifests do snapshot
                 +---------------------------+
                 |  Manifest Files (.avro)    |  <- Lista de data files + stats
                 +---------------------------+
                 |  Data Files (.parquet)     |  <- Dados reais
                 +---------------------------+
```

**Por que Iceberg em vez de Parquet puro?**
- **ACID transactions** — leituras e escritas atomicas
- **Schema evolution** — adicionar/remover/renomear colunas sem reescrever dados
- **Partition evolution** — mudar estrategia de particao sem reescrever dados
- **Time travel** — consultar dados como estavam em qualquer ponto no tempo
- **Hidden partitioning** — usuario nao precisa saber como os dados sao particionados
- **Row-level operations** — UPDATE, DELETE, MERGE eficientes

## 6.2 — Iceberg vs Delta Lake vs Hudi

| Feature | Iceberg | Delta Lake | Hudi |
|---------|---------|-----------|------|
| Governanca | Apache Foundation (aberto) | Databricks (open-source) | Apache Foundation |
| Schema Evolution | Completa (add/drop/rename/reorder) | Add/rename/drop | Add only |
| **Partition Evolution** | **Sim (sem reescrever)** | Nao | Nao |
| **Hidden Partitioning** | **Sim** | Nao | Nao |
| Time Travel | Snapshots | Versoes (log) | Timeline |
| Engines suportados | Spark, Flink, Trino, Dremio, Presto | Spark (nativo), Trino (limitado) | Spark, Flink |
| MERGE INTO | Sim (copy-on-write + merge-on-read) | Sim | Sim |
| Compaction | `rewrite_data_files` | `OPTIMIZE` | Automatico |
| Formato de dados | Parquet, ORC, Avro | Parquet only | Parquet, Avro |

**Quando usar Iceberg:**
- Multi-engine (Spark + Trino + Dremio)
- Precisa de partition evolution
- Tabelas grandes com schema em evolucao
- Quer independencia de vendor

## 6.3 — Configuracao do Iceberg com PySpark

O JAR `iceberg-spark-runtime-3.5_2.12-1.9.1.jar` ja esta instalado nesta stack.

**Tipos de Catalog:**

| Catalog | Metadados em | Quando usar |
|---------|-------------|-------------|
| **Hadoop** | Filesystem (S3/HDFS) | Simples, sem dependencias extras |
| REST | Servidor REST | Multi-engine, centralizado |
| JDBC | Banco relacional | Empresas com PostgreSQL/MySQL |
| Hive | Hive Metastore | Ecossistema Hadoop existente |
| Nessie | Nessie server | Git-like versioning de tabelas |

Nesta stack usaremos **Hadoop catalog** com S3A (MinIO) — nao precisa de servicos extras.

In [ ]:
# Parar a sessao anterior (Iceberg precisa de configs especificas)
spark.stop()

from pyspark.sql import SparkSession

spark_iceberg = SparkSession.builder \
    .appName("agente-iceberg") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.iceberg", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.iceberg.type", "hadoop") \
    .config("spark.sql.catalog.iceberg.warehouse", "s3a://bronze/iceberg-warehouse") \
    .getOrCreate()

print(f"Spark Iceberg session: {spark_iceberg.version}")
print(f"Catalog: iceberg (hadoop)")
print(f"Warehouse: s3a://bronze/iceberg-warehouse")

## 6.4 — CRUD com Iceberg

### Criar tabela e inserir dados

In [ ]:
from pyspark.sql import functions as F
from datetime import date, datetime

# Criar namespace (database)
spark_iceberg.sql("CREATE NAMESPACE IF NOT EXISTS iceberg.engenharia")

# Criar tabela Iceberg via SQL
spark_iceberg.sql("""
    CREATE TABLE IF NOT EXISTS iceberg.engenharia.produtos (
        id          INT,
        nome        STRING,
        categoria   STRING,
        preco       DOUBLE,
        estoque     INT,
        data_cadastro DATE
    )
    USING iceberg
    PARTITIONED BY (categoria)
""")

# Inserir dados via DataFrame
dados_produtos = spark_iceberg.createDataFrame([
    (1, "Notebook Dell", "Eletronicos", 4500.0, 50, date(2024, 1, 15)),
    (2, "Mouse Logitech", "Eletronicos", 150.0, 200, date(2024, 1, 15)),
    (3, "Cadeira Gamer", "Moveis", 1200.0, 30, date(2024, 2, 1)),
    (4, "Mesa Standing", "Moveis", 800.0, 25, date(2024, 2, 10)),
    (5, "Monitor 27pol", "Eletronicos", 2200.0, 80, date(2024, 3, 5)),
    (6, "Teclado Mecanico", "Eletronicos", 350.0, 150, date(2024, 3, 10)),
    (7, "Cadeira Escritorio", "Moveis", 600.0, 45, date(2024, 3, 15)),
    (8, "Webcam HD", "Eletronicos", 280.0, 100, date(2024, 4, 1))
], ["id", "nome", "categoria", "preco", "estoque", "data_cadastro"])

dados_produtos.writeTo("iceberg.engenharia.produtos").append()

# Ler tabela Iceberg
df_prod = spark_iceberg.table("iceberg.engenharia.produtos")
df_prod.show(truncate=False)
print(f"Registros: {df_prod.count()}")

### UPDATE, DELETE e MERGE INTO

In [ ]:
# UPDATE — atualizar registros existentes
spark_iceberg.sql("""
    UPDATE iceberg.engenharia.produtos
    SET preco = preco * 1.10, estoque = estoque - 5
    WHERE categoria = 'Eletronicos'
""")

print("=== Apos UPDATE (10% aumento em Eletronicos) ===")
spark_iceberg.table("iceberg.engenharia.produtos") \
    .filter(F.col("categoria") == "Eletronicos") \
    .select("nome", "preco", "estoque").show()

# DELETE — remover registros
spark_iceberg.sql("""
    DELETE FROM iceberg.engenharia.produtos
    WHERE estoque < 30
""")

print("=== Apos DELETE (estoque < 30) ===")
spark_iceberg.table("iceberg.engenharia.produtos").show()

# MERGE INTO — upsert (atualiza existentes, insere novos)
novos = spark_iceberg.createDataFrame([
    (1, "Notebook Dell XPS", "Eletronicos", 5500.0, 60, date(2024, 5, 1)),   # update
    (9, "Headset Bluetooth", "Eletronicos", 450.0, 120, date(2024, 5, 1)),    # insert
    (10, "Estante Madeira", "Moveis", 950.0, 15, date(2024, 5, 1))            # insert
], ["id", "nome", "categoria", "preco", "estoque", "data_cadastro"])

novos.createOrReplaceTempView("novos_produtos")

spark_iceberg.sql("""
    MERGE INTO iceberg.engenharia.produtos AS target
    USING novos_produtos AS source
    ON target.id = source.id
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")

print("=== Apos MERGE INTO ===")
spark_iceberg.table("iceberg.engenharia.produtos").orderBy("id").show(truncate=False)

## 6.5 — Time Travel (Snapshots)

Cada operacao de escrita cria um **snapshot**. Voce pode consultar qualquer versao anterior dos dados.

In [ ]:
# Ver historico de snapshots
spark_iceberg.sql("SELECT * FROM iceberg.engenharia.produtos.snapshots").show(truncate=False)

# Ver historico de operacoes
spark_iceberg.sql("SELECT * FROM iceberg.engenharia.produtos.history").show(truncate=False)

# Consultar uma versao anterior (snapshot_id)
snapshots = spark_iceberg.sql(
    "SELECT snapshot_id FROM iceberg.engenharia.produtos.snapshots ORDER BY committed_at"
).collect()

if len(snapshots) > 1:
    primeiro_snapshot = snapshots[0]["snapshot_id"]
    print(f"\n=== Dados do primeiro snapshot ({primeiro_snapshot}) ===")
    spark_iceberg.read \
        .option("snapshot-id", primeiro_snapshot) \
        .table("iceberg.engenharia.produtos") \
        .show(truncate=False)

# Consultar por timestamp
# spark_iceberg.read \
#     .option("as-of-timestamp", "1704067200000") \
#     .table("iceberg.engenharia.produtos").show()

# Rollback para snapshot anterior (CUIDADO — irreversivel)
# spark_iceberg.sql(f"""
#     CALL iceberg.system.rollback_to_snapshot('engenharia.produtos', {primeiro_snapshot})
# """)

## 6.6 — Schema Evolution

Iceberg permite alterar o schema **sem reescrever dados**. Os arquivos antigos continuam validos — as novas colunas retornam NULL para dados antigos.

In [ ]:
# Adicionar coluna
spark_iceberg.sql("""
    ALTER TABLE iceberg.engenharia.produtos
    ADD COLUMNS (peso_kg DOUBLE, ativo BOOLEAN)
""")

# Renomear coluna
spark_iceberg.sql("""
    ALTER TABLE iceberg.engenharia.produtos
    RENAME COLUMN preco TO preco_unitario
""")

# Verificar schema atualizado
spark_iceberg.table("iceberg.engenharia.produtos").printSchema()

# Dados antigos tem NULL nas novas colunas
spark_iceberg.table("iceberg.engenharia.produtos") \
    .select("nome", "preco_unitario", "peso_kg", "ativo") \
    .show(3)

# Inserir dados com o novo schema
novos_com_peso = spark_iceberg.createDataFrame([
    (11, "Monitor Curvo 34pol", "Eletronicos", 3800.0, 70, date(2024, 6, 1), 8.5, True),
], ["id", "nome", "categoria", "preco_unitario", "estoque", "data_cadastro", "peso_kg", "ativo"])

novos_com_peso.writeTo("iceberg.engenharia.produtos").append()

spark_iceberg.table("iceberg.engenharia.produtos") \
    .orderBy(F.col("id").desc()) \
    .select("id", "nome", "peso_kg", "ativo") \
    .show(3)

# Dropar coluna (dados da coluna ficam nos arquivos mas nao sao mais lidos)
# spark_iceberg.sql("ALTER TABLE iceberg.engenharia.produtos DROP COLUMN peso_kg")

## 6.7 — Partition Evolution e Hidden Partitioning

**Hidden Partitioning:** O usuario faz queries sem saber como os dados sao particionados. Iceberg aplica os filtros automaticamente.

**Partition Evolution:** Muda a estrategia de particionamento sem reescrever dados antigos. Dados novos usam a nova particao, dados antigos continuam com a anterior.

Funcoes de particao disponiveis:

| Funcao | Exemplo | Resultado |
|--------|---------|----------|
| `year(col)` | `year(data)` | Particiona por ano |
| `month(col)` | `month(data)` | Particiona por ano-mes |
| `day(col)` | `day(data)` | Particiona por ano-mes-dia |
| `hour(col)` | `hour(ts)` | Particiona por ano-mes-dia-hora |
| `bucket(n, col)` | `bucket(16, id)` | Hash em N buckets |
| `truncate(n, col)` | `truncate(3, nome)` | Primeiros N caracteres |

In [ ]:
# Criar tabela com hidden partitioning
spark_iceberg.sql("""
    CREATE TABLE IF NOT EXISTS iceberg.engenharia.vendas (
        id          BIGINT,
        produto     STRING,
        valor       DOUBLE,
        quantidade  INT,
        data_venda  TIMESTAMP
    )
    USING iceberg
    PARTITIONED BY (month(data_venda), bucket(4, produto))
""")

# Inserir dados — o usuario NAO precisa saber sobre particoes
from datetime import datetime

vendas_data = spark_iceberg.createDataFrame([
    (1, "Notebook", 4500.0, 2, datetime(2024, 1, 15, 10, 30)),
    (2, "Mouse", 150.0, 10, datetime(2024, 1, 20, 14, 0)),
    (3, "Teclado", 350.0, 5, datetime(2024, 2, 5, 9, 15)),
    (4, "Monitor", 2200.0, 3, datetime(2024, 2, 10, 16, 45)),
    (5, "Notebook", 4800.0, 1, datetime(2024, 3, 1, 11, 0)),
    (6, "Mouse", 160.0, 8, datetime(2024, 3, 15, 13, 30)),
], ["id", "produto", "valor", "quantidade", "data_venda"])

vendas_data.writeTo("iceberg.engenharia.vendas").append()

# Query normal — Iceberg aplica partition pruning automaticamente
# O usuario nao precisa filtrar por particao manualmente
spark_iceberg.sql("""
    SELECT produto, SUM(valor * quantidade) as total
    FROM iceberg.engenharia.vendas
    WHERE data_venda >= '2024-02-01'
    GROUP BY produto
    ORDER BY total DESC
""").show()

# Ver particoes da tabela
spark_iceberg.sql("SELECT * FROM iceberg.engenharia.vendas.partitions").show(truncate=False)

# Partition Evolution — mudar particao sem reescrever dados antigos
# spark_iceberg.sql("""
#     ALTER TABLE iceberg.engenharia.vendas
#     REPLACE PARTITION FIELD month(data_venda) WITH day(data_venda)
# """)

## 6.8 — Manutencao de Tabelas Iceberg

Tabelas Iceberg acumulam snapshots e arquivos antigos. Manutencao periodica e essencial.

In [ ]:
# Ver metadados da tabela
spark_iceberg.sql("SELECT * FROM iceberg.engenharia.produtos.snapshots").show(truncate=False)
spark_iceberg.sql("SELECT * FROM iceberg.engenharia.produtos.files").show(truncate=False)

# Expire snapshots antigos (libera espaco, perde time travel antigo)
# spark_iceberg.sql("""
#     CALL iceberg.system.expire_snapshots(
#         table => 'engenharia.produtos',
#         older_than => TIMESTAMP '2024-06-01 00:00:00',
#         retain_last => 3
#     )
# """)

# Compaction — junta small files em arquivos maiores
# spark_iceberg.sql("""
#     CALL iceberg.system.rewrite_data_files(
#         table => 'engenharia.produtos',
#         options => map('target-file-size-bytes', '134217728')
#     )
# """)
# target = 128 MB (134217728 bytes)

# Rewrite manifests — otimiza manifest files
# spark_iceberg.sql("""
#     CALL iceberg.system.rewrite_manifests('engenharia.produtos')
# """)

# Remove orphan files — arquivos que nao pertencem a nenhum snapshot
# spark_iceberg.sql("""
#     CALL iceberg.system.remove_orphan_files(
#         table => 'engenharia.produtos',
#         older_than => TIMESTAMP '2024-06-01 00:00:00'
#     )
# """)

print("Procedures de manutencao Iceberg (descomente para executar)")
print("Rotina recomendada:")
print("1. expire_snapshots  -> diario (retain_last=5)")
print("2. rewrite_data_files -> semanal (compaction)")
print("3. rewrite_manifests -> semanal")
print("4. remove_orphan_files -> mensal")

---

# 7. Troubleshooting

## 7.1 — Problemas Comuns e Solucoes

| Problema | Causa | Solucao |
|----------|-------|--------|
| `java.lang.OutOfMemoryError: Java heap space` | Driver sem memoria | Aumentar `spark.driver.memory` |
| `java.lang.OutOfMemoryError: GC overhead` | Executor sem memoria | Aumentar `spark.executor.memory` ou reduzir dados por particao |
| `Container killed by YARN for exceeding memory` | Overhead de memoria | Aumentar `spark.executor.memoryOverhead` (10-20% da executor memory) |
| `Initial job has not accepted any resources` | Cluster sem recursos disponiveis | Reduzir `spark.cores.max` ou `spark.executor.memory`. Verificar se ha apps monopolizando recursos |
| `Connection refused` ao MinIO | MinIO nao subiu ou endpoint errado | Verificar `docker compose ps` e endpoint no spark-defaults.conf |
| `No FileSystem for scheme: s3a` | JAR hadoop-aws ausente | Verificar se JARs estao em `/opt/bitnami/spark/jars/` |
| `Task not serializable` | Objeto nao serializavel no closure | Usar broadcast variables ou mover logica para funcoes serializaveis |
| Shuffle muito lento | Muitas particoes ou data skew | Ajustar `spark.sql.shuffle.partitions`, usar salting ou broadcast join |
| Small files no storage | Muitas particoes na escrita | Usar `coalesce(n)` antes de `write`, ou compaction periodica |
| Iceberg `Table not found` | Namespace/catalog errado | Verificar `spark.sql.catalog.iceberg` e usar nome completo `iceberg.namespace.tabela` |

## 7.2 — Diagnostico Rapido

In [ ]:
# Diagnostico do cluster
print("=== Diagnostico ===")
try:
    sc = spark_iceberg.sparkContext
    print(f"App: {sc.appName}")
    print(f"Master: {sc.master}")
    print(f"Parallelism: {sc.defaultParallelism}")
    print(f"UI: http://localhost:8090")
    print(f"History: http://localhost:18080")
    
    # Testar MinIO
    try:
        test_df = spark_iceberg.read.json("s3a://landing/")
        print(f"MinIO: OK ({test_df.count()} registros no landing)")
    except Exception as e:
        print(f"MinIO: ERRO - {str(e)[:100]}")
    
    # Testar Iceberg
    try:
        spark_iceberg.sql("SHOW NAMESPACES IN iceberg").show()
        print("Iceberg catalog: OK")
    except Exception as e:
        print(f"Iceberg: ERRO - {str(e)[:100]}")
        
except Exception as e:
    print(f"Cluster nao disponivel: {e}")

## 7.3 — Links Uteis

| Recurso | URL |
|---------|-----|
| Spark Master UI | http://localhost:8090 |
| History Server | http://localhost:18080 |
| Jupyter 1 | http://localhost:8888/?token=spark123 |
| Jupyter 2 | http://localhost:8889/?token=spark123 |
| MinIO Console | http://localhost:9001 |
| Dremio | http://localhost:9047 |
| PySpark Docs | https://spark.apache.org/docs/3.5.5/api/python/ |
| Iceberg Docs | https://iceberg.apache.org/docs/latest/ |

In [ ]:
# Encerrar sessao
spark_iceberg.stop()
print("Sessao encerrada")